In [108]:
import pandas as pd
import numpy as np
from pandas.core.methods.selectn import DataFrame

from pandasclean import find_outliers,reduce_memory,handle_nan
import random

ImportError: cannot import name 'handle_nan' from 'auto' (E:\idgaf\DF_Cleaner\auto.py)

In [109]:


def get_customer_data():
    """
    Generates a dataset of 100 customers with numerical and string columns.
    Includes intentional numerical outliers for testing.
    """
    np.random.seed(42)
    random.seed(42) # Ensures consistent random strings

    # String data options
    tiers = ["Bronze", "Silver", "Gold", "Platinum"]
    devices = ["Mobile", "Desktop", "Tablet"]
    countries = ["USA", "Canada", "UK", "Australia", "India"]

    # 1. Generate normal, baseline data for 100 customers
    data = {
        'CustomerID': range(101, 201),
        'Membership_Tier': [random.choice(tiers) for _ in range(100)],
        'Device_Type': [random.choice(devices) for _ in range(100)],
        'Country': [random.choice(countries) for _ in range(100)],
        'Age': np.random.normal(40, 10, 100).astype(int),
        'Annual_Income_USD': np.random.normal(60000, 15000, 100).astype(int),
        'Website_Visits_Per_Month': np.random.normal(15, 5, 100).astype(int),
        'Time_Spent_Mins': np.random.normal(30, 10, 100).round(1),
        'Items_In_Cart': np.random.normal(3, 1.5, 100).astype(int),
        'Total_Purchases_Year': np.random.normal(10, 3, 100).astype(int),
        'Total_Spent_USD': np.random.normal(500, 150, 100).round(2),
        'Satisfaction_Score': np.random.randint(1, 6, 100)
    }

    df = pd.DataFrame(data)

    # Clean up any accidental negative values in numerical columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] < 0, 0)
    df['CustomerID'] = range(101, 201) # Fix IDs back to normal

    # 2. Inject some extreme numerical outliers!
    df.loc[10, 'Age'] = 145                      # Impossible age
    df.loc[25, 'Annual_Income_USD'] = 2500000    # Extreme millionaire income
    df.loc[45, 'Items_In_Cart'] = 450            # Unlikely cart size
    df.loc[70, 'Total_Spent_USD'] = 15000.50     # Whale spender
    df.loc[85, 'Time_Spent_Mins'] = 1440         # Spent exactly 24 hours straight on site

    return df

In [25]:
df = get_customer_data()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                100 non-null    int64  
 1   Membership_Tier           100 non-null    str    
 2   Device_Type               100 non-null    str    
 3   Country                   100 non-null    str    
 4   Age                       100 non-null    int64  
 5   Annual_Income_USD         100 non-null    int64  
 6   Website_Visits_Per_Month  100 non-null    int64  
 7   Time_Spent_Mins           100 non-null    float64
 8   Items_In_Cart             100 non-null    int64  
 9   Total_Purchases_Year      100 non-null    int64  
 10  Total_Spent_USD           100 non-null    float64
 11  Satisfaction_Score        100 non-null    int32  
dtypes: float64(2), int32(1), int64(6), str(3)
memory usage: 9.1 KB


In [26]:
x,y=find_outliers(df,strategy='drop')
x.info()

<class 'pandas.DataFrame'>
Index: 83 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                83 non-null     int64  
 1   Membership_Tier           83 non-null     str    
 2   Device_Type               83 non-null     str    
 3   Country                   83 non-null     str    
 4   Age                       83 non-null     int64  
 5   Annual_Income_USD         83 non-null     int64  
 6   Website_Visits_Per_Month  83 non-null     int64  
 7   Time_Spent_Mins           83 non-null     float64
 8   Items_In_Cart             83 non-null     int64  
 9   Total_Purchases_Year      83 non-null     int64  
 10  Total_Spent_USD           83 non-null     float64
 11  Satisfaction_Score        83 non-null     int32  
dtypes: float64(2), int32(1), int64(6), str(3)
memory usage: 8.1 KB


In [27]:
x

,CustomerID,Membership_Tier,Device_Type,Country,Age,Annual_Income_USD,Website_Visits_Per_Month,Time_Spent_Mins,Items_In_Cart,Total_Purchases_Year,Total_Spent_USD,Satisfaction_Score
0,101,Bronze,Desktop,UK,44,38769,16,21.7,0,12,613.55,5
1,102,Bronze,Tablet,USA,38,53690,17,24.4,2,15,361.68,4
2,103,Gold,Desktop,India,46,54859,20,37.5,3,5,630.44,3
3,104,Silver,Tablet,Canada,55,47965,20,36.1,3,11,703.35,2
4,105,Silver,Desktop,UK,37,57580,8,29.8,2,8,562.02,2
...,...,...,...,...,...,...,...,...,...,...,...,...
95,196,Silver,Tablet,India,25,65779,11,25.3,3,8,453.57,1
96,197,Silver,Mobile,USA,42,46742,19,12.9,1,9,548.92,3
97,198,Platinum,Mobile,India,42,62305,16,43.5,2,7,312.33,5
98,199,Bronze,Mobile,USA,40,60873,19,28.9,1,8,638.60,3


In [28]:
import seaborn as sns


print("Loading Titanic dataset...")
df = sns.load_dataset('titanic')
    # Test your function!
df.info()
df

Loading Titanic dataset...
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [29]:

print("Testing 'cap' strategy...")
cleaned_df, outlier_bounds = find_outliers(df, strategy='cap')
cleaned_df.info()
print("Outlier Bounds Found:")
print(outlier_bounds)
cleaned_df

Testing 'cap' strategy...
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    float64 
 1   pclass       891 non-null    float64 
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    float64 
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(5), int64(1), str(5)
memory usage: 80.7 KB
Outlier Bounds Found:
{'survived': (np.float64(-1.5), np.float64(2.5

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0.0,3.0,male,22.0,1.0,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1.0,1.0,female,38.0,1.0,0,65.6344,C,First,woman,False,C,Cherbourg,yes,False
2,1.0,3.0,female,26.0,0.0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1.0,1.0,female,35.0,1.0,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0.0,3.0,male,35.0,0.0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,2.0,male,27.0,0.0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1.0,1.0,female,19.0,0.0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0.0,3.0,female,NaN,1.0,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1.0,1.0,male,26.0,0.0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [30]:
df[df['age']>64.8125]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
33,0,2,male,66.0,0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
54,0,1,male,65.0,0,1,61.9792,C,First,man,True,B,Cherbourg,no,False
96,0,1,male,71.0,0,0,34.6542,C,First,man,True,A,Cherbourg,no,True
116,0,3,male,70.5,0,0,7.7500,Q,Third,man,True,NaN,Queenstown,no,True
280,0,3,male,65.0,0,0,7.7500,Q,Third,man,True,NaN,Queenstown,no,True
456,0,1,male,65.0,0,0,26.5500,S,First,man,True,E,Southampton,no,True
493,0,1,male,71.0,0,0,49.5042,C,First,man,True,NaN,Cherbourg,no,True
630,1,1,male,80.0,0,0,30.0000,S,First,man,True,A,Southampton,yes,True
672,0,2,male,70.0,0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
745,0,1,male,70.0,1,1,71.0000,S,First,man,True,B,Southampton,no,False


In [31]:
cleaned_df[cleaned_df['age']>=64.8125]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
33,0.0,2.0,male,64.8125,0.0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
54,0.0,1.0,male,64.8125,0.0,1,61.9792,C,First,man,True,B,Cherbourg,no,False
96,0.0,1.0,male,64.8125,0.0,0,34.6542,C,First,man,True,A,Cherbourg,no,True
116,0.0,3.0,male,64.8125,0.0,0,7.7500,Q,Third,man,True,NaN,Queenstown,no,True
280,0.0,3.0,male,64.8125,0.0,0,7.7500,Q,Third,man,True,NaN,Queenstown,no,True
456,0.0,1.0,male,64.8125,0.0,0,26.5500,S,First,man,True,E,Southampton,no,True
493,0.0,1.0,male,64.8125,0.0,0,49.5042,C,First,man,True,NaN,Cherbourg,no,True
630,1.0,1.0,male,64.8125,0.0,0,30.0000,S,First,man,True,A,Southampton,yes,True
672,0.0,2.0,male,64.8125,0.0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
745,0.0,1.0,male,64.8125,1.0,1,65.6344,S,First,man,True,B,Southampton,no,False


In [32]:
print("Testing 'cap' strategy...")
cleaned_df, outlier_bounds = find_outliers(df, strategy='drop')
cleaned_df.info()
print("Outlier Bounds Found:")
print(outlier_bounds)
cleaned_df

Testing 'cap' strategy...
<class 'pandas.DataFrame'>
Index: 571 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     571 non-null    int64   
 1   pclass       571 non-null    int64   
 2   sex          571 non-null    str     
 3   age          571 non-null    float64 
 4   sibsp        571 non-null    int64   
 5   parch        571 non-null    int64   
 6   fare         571 non-null    float64 
 7   embarked     571 non-null    str     
 8   class        571 non-null    category
 9   who          571 non-null    str     
 10  adult_male   571 non-null    bool    
 11  deck         93 non-null     category
 12  embark_town  571 non-null    str     
 13  alive        571 non-null    str     
 14  alone        571 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 56.2 KB
Outlier Bounds Found:
{'survived': (np.float64(-1.5), np.float64(2.5)), '

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [33]:
x=df.memory_usage(deep=True)
print(x.sum())

285564


In [34]:
for i in x:
    print(i)

132
7128
7128
47851
7128
7128
7128
7128
44514
1162
47040
891
1541
52991
45783
891


In [35]:
np.iinfo(np.int8)

iinfo(min=-128, max=127, dtype=int8)

In [36]:
x=df['sibsp'].dtype
print(x)
y=np.iinfo(x)

int64


In [37]:
print(y.min)

-9223372036854775808


In [38]:
df['sibsp'].min()

np.int64(0)

In [39]:
np.finfo(df['age'].dtype)

finfo(resolution=1e-15, min=-1.7976931348623157e+308, max=1.7976931348623157e+308, dtype=float64)

In [40]:
df['embark_town'].max()

'Southampton'

In [41]:
import pandas as pd

# Create a series with repeated strings (low cardinality)
s = pd.Series(['male', 'female', 'male', 'female', 'male'] * 10000)

print("Before conversion:")
print(f"Dtype: {s.dtype}")
print(f"Memory: {s.memory_usage(deep=True)} bytes")

s_category = s.astype('category')

print("\nAfter conversion:")
print(f"Dtype: {s_category.dtype}")
print(f"Memory: {s_category.memory_usage(deep=True)} bytes")

Before conversion:
Dtype: str
Memory: 2690132 bytes

After conversion:
Dtype: category
Memory: 50240 bytes


In [42]:
import numpy as np

# Create a series with almost all unique values (high cardinality)
s2 = pd.Series([f"user_{i}" for i in range(50000)])

print("Before conversion:")
print(f"Memory: {s2.memory_usage(deep=True)} bytes")

s2_category = s2.astype('category')

print("\nAfter conversion:")
print(f"Memory: {s2_category.memory_usage(deep=True)} bytes")

Before conversion:
Memory: 2939022 bytes

After conversion:
Memory: 3139022 bytes


In [43]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 100000

df = pd.DataFrame({
    # Integer columns - different ranges to test downcasting
    'age':        np.random.randint(1, 100, n),           # fits int8
    'year':       np.random.randint(1900, 2024, n),       # fits int16
    'salary':     np.random.randint(20000, 200000, n),    # fits int32
'big_number': np.random.randint(3000000000, 3000100000, n, dtype=np.int64),
    # Float columns
    'score':      np.random.uniform(0, 100, n),           # fits float32
    'ratio':      np.random.uniform(0, 1, n),             # fits float32

    # Object columns - low cardinality (should convert to category)
    'gender':     np.random.choice(['Male', 'Female', 'Other'], n),
    'status':     np.random.choice(['Active', 'Inactive', 'Pending'], n),
    'department': np.random.choice(['HR', 'IT', 'Finance', 'Marketing'], n),

    # Object columns - high cardinality (should NOT convert)
    'user_id':    [f'user_{i}' for i in range(n)],
    'email':      [f'user_{i}@email.com' for i in range(n)],
})

print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')
x,y=reduce_memory(df)
print('after')
print(x.dtypes)
print(f'\nMemory usage: {x.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB')

age             int32
year            int32
salary          int32
big_number      int64
score         float64
ratio         float64
gender            str
status            str
department        str
user_id           str
email             str
dtype: object

Memory usage: 31.26 MB
31.26-->14.67 (16.59mb saved)
after
age               int8
year             int16
salary           int32
big_number       int64
score          float32
ratio          float32
gender        category
status        category
department    category
user_id            str
email              str
dtype: object

Memory usage: 14.67 MB


In [44]:
for col in ['gender', 'status', 'department', 'user_id']:
    print(f"{col}: {df[col].dtype} | is_object: {pd.api.types.is_object_dtype(df[col])}")

gender: str | is_object: False
status: str | is_object: False
department: str | is_object: False
user_id: str | is_object: False


In [45]:
for col in ['gender', 'user_id']:
    print(f"{col}: {df[col].dtype}")
    print(f"is_object: {pd.api.types.is_object_dtype(df[col])}")
    print(f"is_string: {pd.api.types.is_string_dtype(df[col])}")

gender: str
is_object: False
is_string: True
user_id: str
is_object: False
is_string: True


In [89]:
import pandas as pd
import numpy as np

def get_massive_dataframe(num_rows=1_500_000):
    """
    Generates a DataFrame with 1.5 million rows to safely exceed 100MB in memory.
    """
    print(f"Generating massive DataFrame with {num_rows:,} rows. Please wait a few seconds...")

    # Setting a seed so the random data is the same every time you run it
    np.random.seed(42)

    # 1. Generate millions of data points using fast numpy arrays
    data = {
        'Account_ID': np.arange(num_rows),
        'Customer_Age': np.random.randint(18, 95, size=num_rows),
        'Account_Balance': np.random.normal(5000, 2000, size=num_rows),
        'Monthly_Spend': np.random.exponential(500, size=num_rows),
        'Credit_Score': np.random.normal(650, 100, size=num_rows),
        'Activity_Metric_1': np.random.uniform(0, 100, size=num_rows),
        'Activity_Metric_2': np.random.uniform(0, 100, size=num_rows),
        # Adding a string category forces pandas to use a lot of memory overhead
        'Tier': np.random.choice(['Standard', 'Premium', 'VIP', 'Elite'], size=num_rows)
    }

    df = pd.DataFrame(data)

    # 2. Calculate and print the exact memory footprint
    # 'deep=True' forces pandas to accurately weigh the string column as well
    memory_in_bytes = df.memory_usage(deep=True).sum()
    memory_in_mb = memory_in_bytes / (1024 ** 2)

    print("-" * 30)
    print(f"Data Generation Complete!")
    print(f"Total Memory Usage: {memory_in_mb:.2f} MB")
    print("-" * 30)

    return df

In [95]:
df=get_massive_dataframe(100000)
df.info()
df.to_csv(r'D:\before.csv', index=False)

x,y=reduce_memory(df)
x.info()
x.to_csv(r'D:\after.csv', index=False)

Generating massive DataFrame with 100,000 rows. Please wait a few seconds...
------------------------------
Data Generation Complete!
Total Memory Usage: 10.18 MB
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Account_ID         100000 non-null  int64  
 1   Customer_Age       100000 non-null  int32  
 2   Account_Balance    100000 non-null  float64
 3   Monthly_Spend      100000 non-null  float64
 4   Credit_Score       100000 non-null  float64
 5   Activity_Metric_1  100000 non-null  float64
 6   Activity_Metric_2  100000 non-null  float64
 7   Tier               100000 non-null  str    
dtypes: float64(5), int32(1), int64(1), str(1)
memory usage: 5.7 MB


ModuleNotFoundError: No module named 'openpyxl'

In [96]:
import pandas as pd
import numpy as np

def get_messy_dataset():
    """
    Generates a 10-row dataset with various types of missing data (NaN, None, empty strings)
    perfect for testing dropna() and fillna() logic.
    """
    data = {
        'CustomerID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], # Perfect column, no missing data

        # 1. Standard float NaNs (Very common)
        'Age': [25, np.nan, 30, 45, np.nan, 22, 34, 50, 29, 40],
        'Account_Balance': [1500.50, 200.00, np.nan, 8500.00, 300.75, np.nan, 0.00, 450.10, np.nan, 1000.00],

        # 2. String/Categorical missing data (Using None)
        'Name': ['Alice', 'Bob', None, 'David', 'Eve', 'Frank', None, 'Hannah', 'Ian', 'Jane'],

        # 3. The "Silent" Missing Data (Empty strings)
        # Pandas does NOT automatically consider "" as a NaN!
        'Region': ['North', 'South', 'East', 'West', None, 'North', 'South', 'East', '', 'West'],

        # 4. Boolean missing data
        'Is_Active': [True, False, np.nan, True, True, False, False, np.nan, True, True]
    }

    df = pd.DataFrame(data)
    return df

In [97]:
df=get_messy_dataset()
df

,CustomerID,Age,Account_Balance,Name,Region,Is_Active
0,1,25.0,1500.50,Alice,North,True
1,2,NaN,200.00,Bob,South,False
2,3,30.0,NaN,NaN,East,NaN
3,4,45.0,8500.00,David,West,True
4,5,NaN,300.75,Eve,NaN,True
5,6,22.0,NaN,Frank,North,False
6,7,34.0,0.00,NaN,South,False
7,8,50.0,450.10,Hannah,East,NaN
8,9,29.0,NaN,Ian,,True
9,10,40.0,1000.00,Jane,West,True


In [98]:
len(df["Age"])-df['Age'].isna().sum()

np.int64(8)

In [99]:
df.dropna()

,CustomerID,Age,Account_Balance,Name,Region,Is_Active
0,1,25.0,1500.5,Alice,North,True
3,4,45.0,8500.0,David,West,True
9,10,40.0,1000.0,Jane,West,True


In [102]:
len(df.columns)

6

In [105]:
df['Age'].mean()

np.float64(34.375)